In [ ]:
import numpy as np
import sqlite3
conn = sqlite3.connect("Rock_data.db")
pointer=conn.cursor()



import csv


pointer.execute("""
CREATE TABLE IF NOT EXISTS Table3_rock_data (
    Month TEXT,
    Price DECIMAL(10, 2),
    Volume NUMERIC,
    Net_Foreign_Flow NUMERIC
)
""")

with open("../Resources/nse_analysis_20261016_103929.csv", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        Date = row["Date"]      # pick specific CSV columns
        price = row["Price (KES)"]
        Volume = row["Volume"]
        Net_Foreign_Flow = row["Net Foreign Flow"]

        Price=price.strip('KES ')
        
        Net_Foreign_Flow=Net_Foreign_Flow.strip()

        pointer.execute(
            "INSERT INTO Table_rock_data (Date, Price, Volume, Net_Foreign_Flow) VALUES (?, ?, ?, ?)",
            (Date, Price, Volume, Net_Foreign_Flow)
        )

conn.commit()



year=['2023','2024','2025','2026']

months=['1','2','3','4','5','6','7','8','9','10','11','12']

for i in year:
    for j in months:
        pointer.execute("""
SELECT * FROM Table_rock_data
WHERE Date LIKE '""" + j + """/%/""" + i + """'
""")
        
        rows = pointer.fetchall()
        sumo=[]
        for row in rows:
            try:
                q=float(row[1])
                if q==0:
                    continue
            except (ValueError, TypeError):
                continue
            sumo.append(row[1])
        p=np.average(sumo)
        sumo=[]
        for row in rows:
            try:
                q=int(row[2])
                if q==0:
                    continue
            except (ValueError, TypeError):
                continue
            sumo.append(row[2])
        v=(np.sum(sumo))
        sumo=[]
        for row in rows:
            try:
                q=int(row[3])
                if q==0:
                    continue
            except (ValueError, TypeError):
                continue
                
            sumo.append(row[3])
        n=(np.sum(sumo))

        Net_Foreign_Flow=int(n)
        Month=str(" " + j + "/" + i)
        Volume=int(v)
        Price=np.round(p,2)
        pointer.execute(
            "INSERT INTO Table2_rock_data (Month, Price, Volume, Net_Foreign_Flow) VALUES (?, ?, ?, ?)",
            (Month, Price, Volume, Net_Foreign_Flow)
        )
        conn.commit()


In [9]:
import numpy as np
import sqlite3
conn = sqlite3.connect("Rock_data.db")
pointer=conn.cursor()



import csv


pointer.execute("""
CREATE TABLE IF NOT EXISTS Table4a_rock_data (
    Month TEXT,
    Exchange_Rate DECIMAL(10, 2))
""")

with open("../Resources/exchange_rates_2026-10-18.csv", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        Month = row["Date"]      # pick specific CSV columns
        Exchange_Rate = row["Exchange Rate"]
        

        pointer.execute(
            "INSERT INTO Table4a_rock_data (Month, Exchange_Rate) VALUES (?, ?)",
            (Month, Exchange_Rate)
        )

        conn.commit()


'''
year=['2023','2024','2025','2026']

year=['2026']
months=['1','2','3','4','5','6','7','8','9','10','11','12']

months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
for i in year:
    for j in months:
        pointer.execute("""
SELECT * FROM Table4a_rock_data
WHERE Month LIKE '""" + j + """/%/""" + i + """'
""")
        
        rows = pointer.fetchall()
        sumo=[]
        for row in rows:
            sumo.append(row[2])
        p=np.average(sumo)
        
        

        Month=str(" " + j + "/" + i)
        
        Exchange_Rate =np.round(p,2)

        pointer.execute("CREATE TABLE IF NOT EXISTS Table4b_rock_data (Month TEXT, Exchange_Rate DECIMAL(10, 2))")
        conn.commit()
        pointer.execute(
            "INSERT INTO Table4b_rock_data (Month, Exchange_Rate) VALUES (?, ?)",
            (Month, Exchange_Rate)
        )
        conn.commit()

'''

'\nyear=[\'2023\',\'2024\',\'2025\',\'2026\']\n\nyear=[\'2026\']\nmonths=[\'1\',\'2\',\'3\',\'4\',\'5\',\'6\',\'7\',\'8\',\'9\',\'10\',\'11\',\'12\']\n\nmonths=[\'Jan\',\'Feb\',\'Mar\',\'Apr\',\'May\',\'Jun\',\'Jul\',\'Aug\',\'Sep\',\'Oct\',\'Nov\',\'Dec\']\nfor i in year:\n    for j in months:\n        pointer.execute("""\nSELECT * FROM Table4a_rock_data\nWHERE Month LIKE \'""" + j + """/%/""" + i + """\'\n""")\n        \n        rows = pointer.fetchall()\n        sumo=[]\n        for row in rows:\n            sumo.append(row[2])\n        p=np.average(sumo)\n        \n        \n\n        Month=str(" " + j + "/" + i)\n        \n        Exchange_Rate =np.round(p,2)\n\n        pointer.execute("CREATE TABLE IF NOT EXISTS Table4b_rock_data (Month TEXT, Exchange_Rate DECIMAL(10, 2))")\n        conn.commit()\n        pointer.execute(\n            "INSERT INTO Table4b_rock_data (Month, Exchange_Rate) VALUES (?, ?)",\n            (Month, Exchange_Rate)\n        )\n        conn.commit()\n\n'

In [ ]:
"""
Exchange_rate_data.db pipeline
==============================
Stage 1 : load raw daily exchange-rate CSVs -> Table_exchange_rate_daily
Stage 2 : aggregate monthly averages        -> Table_exchange_rate_monthly
Stage 3 : export monthly table              -> ../Results/cleaned_exchange_rates_monthly.csv
"""

import csv
import glob
import os
import sqlite3
from datetime import datetime

DB_PATH = "Exchange_rate_data.db"
RAW_CSV_FOLDER = "../Resources"
RAW_CSV_PATTERN = os.path.join(RAW_CSV_FOLDER, "*.csv")
OUT_CSV = "../Results/cleaned_exchange_rates_monthly.csv"

# Set to "USD/KES" if your files contain several currency pairs.
# Set to None to include every currency pair found in the CSVs.
CURRENCY_PAIR = None


# ----------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------
def normalize_date(raw):
    """
    Converts common date formats to ISO format: YYYY-MM-DD.
    Returns None if the date cannot be parsed.
    """
    if raw is None:
        return None

    raw = str(raw).strip()

    for fmt in ("%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(raw, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue

    return None


def clean_exchange_rate(raw):
    """
    Converts 'KES 123.45', '1,234.56', '123.45' to float.
    Returns None for missing or invalid values.
    """
    if raw is None:
        return None

    s = (
        str(raw)
        .replace("KES", "")
        .replace(",", "")
        .strip()
    )

    if s == "" or s.lower() in {"nan", "none", "n/a", "-"}:
        return None

    try:
        value = float(s)
    except ValueError:
        return None

    return value if value > 0 else None


# ----------------------------------------------------------------------
# Stage 1 : load all raw daily CSV files
# ----------------------------------------------------------------------
def load_raw(cursor, conn):
    cursor.execute("DROP TABLE IF EXISTS Table_exchange_rate_daily")

    cursor.execute("""
        CREATE TABLE Table_exchange_rate_daily (
            Date TEXT PRIMARY KEY,
            Currency_Pair TEXT,
            Exchange_Rate REAL
        )
    """)

    csv_files = sorted(glob.glob(RAW_CSV_PATTERN))

    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files found using pattern: {RAW_CSV_PATTERN}"
        )

    print(f"Found {len(csv_files)} CSV files:")
    for file_path in csv_files:
        print(f"  {file_path}")

    batch = []
    skipped = 0

    for file_path in csv_files:
        with open(file_path, newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)

            for row in reader:
                date = normalize_date(row["Date"])
                rate = clean_exchange_rate(row["Exchange Rate"])
                pair = str(row["Currency_pair"]).strip()

                if date is None or rate is None:
                    skipped += 1
                    continue

                if CURRENCY_PAIR and pair.upper() != CURRENCY_PAIR.upper():
                    continue

                batch.append((date, pair, rate))

    cursor.executemany(
        """
        INSERT OR REPLACE INTO Table_exchange_rate_daily
        (Date, Currency_Pair, Exchange_Rate)
        VALUES (?, ?, ?)
        """,
        batch,
    )

    conn.commit()

    print(f"Inserted {len(batch)} valid daily rows")
    print(f"Skipped {skipped} invalid rows")


# ----------------------------------------------------------------------
# Stage 2 : calculate monthly averages
# ----------------------------------------------------------------------
def aggregate_monthly(cursor, conn):
    cursor.execute("DROP TABLE IF EXISTS Table_exchange_rate_monthly")

    cursor.execute("""
        CREATE TABLE Table_exchange_rate_monthly (
            Month TEXT PRIMARY KEY,
            Exchange_Rate REAL
        )
    """)

    cursor.execute("""
        INSERT INTO Table_exchange_rate_monthly
            (Month, Exchange_Rate)
        SELECT
            substr(Date, 1, 7) AS Month,
            ROUND(AVG(Exchange_Rate), 4) AS Exchange_Rate
        FROM Table_exchange_rate_daily
        GROUP BY substr(Date, 1, 7)
        ORDER BY Month
    """)

    conn.commit()

    cursor.execute(
        "SELECT Month, Exchange_Rate FROM Table_exchange_rate_monthly"
    )

    for month, rate in cursor.fetchall():
        print(f"  {month}: {rate}")


# ----------------------------------------------------------------------
# Stage 3 : export monthly averages
# ----------------------------------------------------------------------
def export_monthly(cursor):
    cursor.execute("""
        SELECT Month, Exchange_Rate
        FROM Table_exchange_rate_monthly
        ORDER BY Month
    """)

    rows = cursor.fetchall()
    column_names = [description[0] for description in cursor.description]

    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(column_names)
        writer.writerows(rows)

    print(f"Wrote {len(rows)} monthly rows to {OUT_CSV}")


# ----------------------------------------------------------------------
# main
# ----------------------------------------------------------------------
def main():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    try:
        load_raw(cursor, conn)
        aggregate_monthly(cursor, conn)
        export_monthly(cursor)
    finally:
        conn.close()
        print("Database connection closed.")


if __name__ == "__main__":
    main()

In [10]:
"""
Exchange_rate_data.db pipeline
==============================
Stage 1 : load raw daily exchange-rate CSVs -> Table_exchange_rate_daily
Stage 2 : aggregate monthly averages        -> Table_exchange_rate_monthly
Stage 3 : export monthly table              -> ../Results/cleaned_exchange_rates_monthly.csv

Expected raw CSV headers:
    "Date","Currency_pair","Exchange Rate",

Expected date format:
    Dec 31, 2024
"""

import csv
import glob
import os
import sqlite3
from datetime import datetime

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
DB_PATH = "Exchange_rate_data.db"
RAW_CSV_FOLDER = "../Resources"
RAW_CSV_PATTERN = os.path.join(RAW_CSV_FOLDER, "*.csv")
OUT_CSV = "../Results/cleaned_exchange_rates_monthly.csv"

# Use "USD/KES" to process only that currency pair.
# Use None only if each CSV contains one currency pair and you want all of them combined.
CURRENCY_PAIR = "USD/KES"


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def normalize_date(raw):
    """
    Converts dates such as 'Dec 31, 2024' to ISO format: '2024-12-31'.
    Returns None if the date cannot be parsed.
    """
    if raw is None:
        return None

    raw = str(raw).strip()

    date_formats = (
        "%b %d, %Y",     # Dec 31, 2024
        "%B %d, %Y",     # December 31, 2024
        "%m/%d/%Y",      # 12/31/2024
        "%d/%m/%Y",      # 31/12/2024
        "%Y-%m-%d",      # 2024-12-31
    )

    for fmt in date_formats:
        try:
            return datetime.strptime(raw, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue

    return None


def clean_exchange_rate(raw):
    """
    Converts exchange-rate values such as '129.2927' or '1,234.56' to float.
    Returns None for missing, invalid, zero, or negative values.
    """
    if raw is None:
        return None

    s = str(raw).replace(",", "").strip()

    if s == "" or s.lower() in {"nan", "none", "n/a", "-"}:
        return None

    try:
        value = float(s)
    except ValueError:
        return None

    return value if value > 0 else None


# ----------------------------------------------------------------------
# Stage 1 : Load all raw daily CSV files
# ----------------------------------------------------------------------
def load_raw(cursor, conn):
    """
    Reads every CSV file matching RAW_CSV_PATTERN,
    cleans the records, and loads them into SQLite.
    """
    cursor.execute("DROP TABLE IF EXISTS Table_exchange_rate_daily")

    cursor.execute("""
        CREATE TABLE Table_exchange_rate_daily (
            Date TEXT PRIMARY KEY,
            Currency_Pair TEXT,
            Exchange_Rate REAL
        )
    """)

    csv_files = sorted(glob.glob(RAW_CSV_PATTERN))

    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files found using pattern: {RAW_CSV_PATTERN}"
        )

    print(f"Found {len(csv_files)} CSV files:")
    for file_path in csv_files:
        print(f"  {file_path}")

    batch = []
    skipped_invalid = 0
    skipped_other_pair = 0

    for file_path in csv_files:
        with open(file_path, newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)

            for row in reader:
                date = normalize_date(row.get("Date"))
                pair = str(row.get("Currency_pair", "")).strip()
                rate = clean_exchange_rate(row.get("Exchange Rate"))

                if date is None or rate is None:
                    skipped_invalid += 1
                    continue

                if CURRENCY_PAIR and pair.upper() != CURRENCY_PAIR.upper():
                    skipped_other_pair += 1
                    continue

                batch.append((
                    date,
                    pair,
                    rate,
                ))

    cursor.executemany(
        """
        INSERT OR REPLACE INTO Table_exchange_rate_daily
            (Date, Currency_Pair, Exchange_Rate)
        VALUES (?, ?, ?)
        """,
        batch,
    )

    conn.commit()

    print(f"Inserted {len(batch)} valid daily rows into Table_exchange_rate_daily")
    print(f"Skipped {skipped_invalid} rows with invalid dates or rates")

    if CURRENCY_PAIR:
        print(f"Skipped {skipped_other_pair} rows for other currency pairs")


# ----------------------------------------------------------------------
# Stage 2 : Calculate monthly average exchange rates
# ----------------------------------------------------------------------
def aggregate_monthly(cursor, conn):
    """
    Groups daily exchange rates by YYYY-MM and calculates the mon

_IncompleteInputError: incomplete input (2445275255.py, line 163)

In [11]:
"""
Exchange_rate_data.db pipeline
==============================
Stage 1 : load raw daily exchange-rate CSVs -> Table_exchange_rate_daily
Stage 2 : aggregate monthly averages        -> Table_exchange_rate_monthly
Stage 3 : export monthly table              -> ../Results/cleaned_exchange_rates_monthly.csv

Expected raw CSV headers:
    "Date","Currency_pair","Exchange Rate",

Expected date format:
    Dec 31, 2024
"""

import csv
import glob
import os
import sqlite3
from datetime import datetime

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
DB_PATH = "Exchange_rate_data.db"
RAW_CSV_FOLDER = "../Resources"
RAW_CSV_PATTERN = os.path.join(RAW_CSV_FOLDER, "*.csv")
OUT_CSV = "../Results/cleaned_exchange_rates_monthly.csv"

# Use "USD/KES" to process only that currency pair.
# Use None only if each CSV contains one currency pair and you want all of them combined.
CURRENCY_PAIR = "USD/KES"


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def normalize_date(raw):
    """
    Converts dates such as 'Dec 31, 2024' to ISO format: '2024-12-31'.
    Returns None if the date cannot be parsed.
    """
    if raw is None:
        return None

    raw = str(raw).strip()

    date_formats = (
        "%b %d, %Y",     # Dec 31, 2024
        "%B %d, %Y",     # December 31, 2024
        "%m/%d/%Y",      # 12/31/2024
        "%d/%m/%Y",      # 31/12/2024
        "%Y-%m-%d",      # 2024-12-31
    )

    for fmt in date_formats:
        try:
            return datetime.strptime(raw, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue

    return None


def clean_exchange_rate(raw):
    """
    Converts exchange-rate values such as '129.2927' or '1,234.56' to float.
    Returns None for missing, invalid, zero, or negative values.
    """
    if raw is None:
        return None

    s = str(raw).replace(",", "").strip()

    if s == "" or s.lower() in {"nan", "none", "n/a", "-"}:
        return None

    try:
        value = float(s)
    except ValueError:
        return None

    return value if value > 0 else None


# ----------------------------------------------------------------------
# Stage 1 : Load all raw daily CSV files
# ----------------------------------------------------------------------
def load_raw(cursor, conn):
    """
    Reads every CSV file matching RAW_CSV_PATTERN,
    cleans the records, and loads them into SQLite.
    """
    cursor.execute("DROP TABLE IF EXISTS Table_exchange_rate_daily")

    cursor.execute("""
        CREATE TABLE Table_exchange_rate_daily (
            Date TEXT PRIMARY KEY,
            Currency_Pair TEXT,
            Exchange_Rate REAL
        )
    """)

    csv_files = sorted(glob.glob(RAW_CSV_PATTERN))

    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files found using pattern: {RAW_CSV_PATTERN}"
        )

    print(f"Found {len(csv_files)} CSV files:")
    for file_path in csv_files:
        print(f"  {file_path}")

    batch = []
    skipped_invalid = 0
    skipped_other_pair = 0

    for file_path in csv_files:
        with open(file_path, newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)

            for row in reader:
                date = normalize_date(row.get("Date"))
                pair = str(row.get("Currency_pair", "")).strip()
                rate = clean_exchange_rate(row.get("Exchange Rate"))

                if date is None or rate is None:
                    skipped_invalid += 1
                    continue

                if CURRENCY_PAIR and pair.upper() != CURRENCY_PAIR.upper():
                    skipped_other_pair += 1
                    continue

                batch.append((
                    date,
                    pair,
                    rate,
                ))

    cursor.executemany(
        """
        INSERT OR REPLACE INTO Table_exchange_rate_daily
            (Date, Currency_Pair, Exchange_Rate)
        VALUES (?, ?, ?)
        """,
        batch,
    )

    conn.commit()

    print(f"Inserted {len(batch)} valid daily rows into Table_exchange_rate_daily")
    print(f"Skipped {skipped_invalid} rows with invalid dates or rates")

    if CURRENCY_PAIR:
        print(f"Skipped {skipped_other_pair} rows for other currency pairs")


# ----------------------------------------------------------------------
# Stage 2 : Calculate monthly average exchange rates
# ----------------------------------------------------------------------
def aggregate_monthly(cursor, conn):
    """
    Groups daily exchange rates by YYYY-MM and calculates the monthly average.
    """
    cursor.execute("DROP TABLE IF EXISTS Table_exchange_rate_monthly")

    cursor.execute("""
        CREATE TABLE Table_exchange_rate_monthly (
            Month TEXT PRIMARY KEY,
            Exchange_Rate REAL
        )
    """)

    cursor.execute("""
        INSERT INTO Table_exchange_rate_monthly
            (Month, Exchange_Rate)
        SELECT
            substr(Date, 1, 7) AS Month,
            ROUND(AVG(Exchange_Rate), 4) AS Exchange_Rate
        FROM Table_exchange_rate_daily
        GROUP BY substr(Date, 1, 7)
        ORDER BY Month
    """)

    conn.commit()

    print("Monthly aggregation complete:")
    print("  Month      Average Exchange Rate")

    cursor.execute("""
        SELECT Month, Exchange_Rate
        FROM Table_exchange_rate_monthly
        ORDER BY Month
    """)

    for month, rate in cursor.fetchall():
        print(f"  {month}   {rate}")


# ----------------------------------------------------------------------
# Stage 3 : Export monthly averages to CSV
# ----------------------------------------------------------------------
def export_monthly(cursor):
    """
    Exports the monthly average exchange rates to a cleaned CSV file.
    """
    cursor.execute("""
        SELECT Month, Exchange_Rate
        FROM Table_exchange_rate_monthly
        ORDER BY Month
    """)

    rows = cursor.fetchall()
    column_names = [description[0] for description in cursor.description]

    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(column_names)
        writer.writerows(rows)

    print(f"Wrote {len(rows)} monthly rows to {OUT_CSV}")


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
def main():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"Connected to database: {DB_PATH}")

    try:
        load_raw(cursor, conn)
        aggregate_monthly(cursor, conn)
        export_monthly(cursor)
    finally:
        conn.close()
        print("Database connection closed.")


if __name__ == "__main__":
    main()

Connected to database: Exchange_rate_data.db
Found 7 CSV files:
  ../Resources\exchange_rates_2026-10-18(1).csv
  ../Resources\exchange_rates_2026-10-18(2).csv
  ../Resources\exchange_rates_2026-10-18(3).csv
  ../Resources\exchange_rates_2026-10-18.csv
  ../Resources\inflation_data_2026-10-16.csv
  ../Resources\monetary_policy_indicators_2026-10-18.csv
  ../Resources\nse_analysis_20261016_103929.csv
Inserted 1000 valid daily rows into Table_exchange_rate_daily
Skipped 1135 rows with invalid dates or rates
Skipped 0 rows for other currency pairs
Monthly aggregation complete:
  Month      Average Exchange Rate
  2023-05   138.3971
  2023-06   139.7493
  2023-07   141.5124
  2023-08   144.0414
  2023-09   146.8216
  2023-10   149.4429
  2023-11   152.1154
  2023-12   154.1563
  2024-01   159.7283
  2024-02   151.2907
  2024-03   137.0702
  2024-04   131.5728
  2024-05   131.6356
  2024-06   129.3439
  2024-07   129.9046
  2024-08   129.3282
  2024-09   129.2019
  2024-10   129.1962
  2024

In [12]:
"""
Exchange_rate_data.db pipeline
==============================
Stage 1 : load raw daily exchange-rate CSVs -> Table_exchange_rate_daily
Stage 2 : aggregate monthly averages        -> Table_exchange_rate_monthly
Stage 3 : export monthly table              -> ../Results/cleaned_exchange_rates_monthly.csv

Expected raw CSV headers:
    "Date","Currency_pair","Exchange Rate",

Expected date format:
    Dec 31, 2024
"""

import csv
import os
import sqlite3
from datetime import datetime

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
DB_PATH = "Exchange_rate_data.db"

# Add your CSV files here. The script will process only these files.
RAW_CSV_FILES = [
    "../Resources/exchange_rates_2026-10-18.csv",
    "../Resources/exchange_rates_2026-10-18(1).csv",
    "../Resources/exchange_rates_2026-10-18(2).csv",
    "../Resources/exchange_rates_2026-10-18(3).csv",
]

OUT_CSV = "../Results/cleaned_exchange_rates_monthly.csv"

# Use "USD/KES" to process only that currency pair.
# Use None only if each CSV contains one currency pair and you want all of them combined.
CURRENCY_PAIR = "USD/KES"


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def normalize_date(raw):
    """
    Converts dates such as 'Dec 31, 2024' to ISO format: '2024-12-31'.
    Returns None if the date cannot be parsed.
    """
    if raw is None:
        return None

    raw = str(raw).strip()

    date_formats = (
        "%b %d, %Y",     # Dec 31, 2024
        "%B %d, %Y",     # December 31, 2024
        "%m/%d/%Y",      # 12/31/2024
        "%d/%m/%Y",      # 31/12/2024
        "%Y-%m-%d",      # 2024-12-31
    )

    for fmt in date_formats:
        try:
            return datetime.strptime(raw, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue

    return None


def clean_exchange_rate(raw):
    """
    Converts exchange-rate values such as '129.2927' or '1,234.56' to float.
    Returns None for missing, invalid, zero, or negative values.
    """
    if raw is None:
        return None

    s = str(raw).replace(",", "").strip()

    if s == "" or s.lower() in {"nan", "none", "n/a", "-"}:
        return None

    try:
        value = float(s)
    except ValueError:
        return None

    return value if value > 0 else None


# ----------------------------------------------------------------------
# Stage 1 : Load the listed raw daily CSV files
# ----------------------------------------------------------------------
def load_raw(cursor, conn):
    """
    Reads the explicitly listed CSV files,
    cleans the records, and loads them into SQLite.
    """
    cursor.execute("DROP TABLE IF EXISTS Table_exchange_rate_daily")

    cursor.execute("""
        CREATE TABLE Table_exchange_rate_daily (
            Date TEXT PRIMARY KEY,
            Currency_Pair TEXT,
            Exchange_Rate REAL
        )
    """)

    if not RAW_CSV_FILES:
        raise ValueError("No CSV files were listed in RAW_CSV_FILES.")

    print(f"Processing {len(RAW_CSV_FILES)} CSV files:")
    for file_path in RAW_CSV_FILES:
        print(f"  {file_path}")

    batch = []
    skipped_invalid = 0
    skipped_other_pair = 0

    for file_path in RAW_CSV_FILES:
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"CSV file not found: {file_path}")

        with open(file_path, newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)

            for row in reader:
                date = normalize_date(row.get("Date"))
                pair = str(row.get("Currency_pair", "")).strip()
                rate = clean_exchange_rate(row.get("Exchange Rate"))

                if date is None or rate is None:
                    skipped_invalid += 1
                    continue

                if CURRENCY_PAIR and pair.upper() != CURRENCY_PAIR.upper():
                    skipped_other_pair += 1
                    continue

                batch.append((
                    date,
                    pair,
                    rate,
                ))

    cursor.executemany(
        """
        INSERT OR REPLACE INTO Table_exchange_rate_daily
            (Date, Currency_Pair, Exchange_Rate)
        VALUES (?, ?, ?)
        """,
        batch,
    )

    conn.commit()

    print(f"Inserted {len(batch)} valid daily rows into Table_exchange_rate_daily")
    print(f"Skipped {skipped_invalid} rows with invalid dates or rates")

    if CURRENCY_PAIR:
        print(f"Skipped {skipped_other_pair} rows for other currency pairs")


# ----------------------------------------------------------------------
# Stage 2 : Calculate monthly average exchange rates
# ----------------------------------------------------------------------
def aggregate_monthly(cursor, conn):
    """
    Groups daily exchange rates by YYYY-MM and calculates the monthly average.
    """
    cursor.execute("DROP TABLE IF EXISTS Table_exchange_rate_monthly")

    cursor.execute("""
        CREATE TABLE Table_exchange_rate_monthly (
            Month TEXT PRIMARY KEY,
            Exchange_Rate REAL
        )
    """)

    cursor.execute("""
        INSERT INTO Table_exchange_rate_monthly
            (Month, Exchange_Rate)
        SELECT
            substr(Date, 1, 7) AS Month,
            ROUND(AVG(Exchange_Rate), 4) AS Exchange_Rate
        FROM Table_exchange_rate_daily
        GROUP BY substr(Date, 1, 7)
        ORDER BY Month
    """)

    conn.commit()

    print("Monthly aggregation complete:")
    print("  Month      Average Exchange Rate")

    cursor.execute("""
        SELECT Month, Exchange_Rate
        FROM Table_exchange_rate_monthly
        ORDER BY Month
    """)

    for month, rate in cursor.fetchall():
        print(f"  {month}   {rate}")


# ----------------------------------------------------------------------
# Stage 3 : Export monthly averages to CSV
# ----------------------------------------------------------------------
def export_monthly(cursor):
    """
    Exports the monthly average exchange rates to a cleaned CSV file.
    """
    cursor.execute("""
        SELECT Month, Exchange_Rate
        FROM Table_exchange_rate_monthly
        ORDER BY Month
    """)

    rows = cursor.fetchall()
    column_names = [description[0] for description in cursor.description]

    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(column_names)
        writer.writerows(rows)

    print(f"Wrote {len(rows)} monthly rows to {OUT_CSV}")


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
def main():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"Connected to database: {DB_PATH}")

    try:
        load_raw(cursor, conn)
        aggregate_monthly(cursor, conn)
        export_monthly(cursor)
    finally:
        conn.close()
        print("Database connection closed.")


if __name__ == "__main__":
    main()

Connected to database: Exchange_rate_data.db
Processing 4 CSV files:
  ../Resources/exchange_rates_2026-10-18.csv
  ../Resources/exchange_rates_2026-10-18(1).csv
  ../Resources/exchange_rates_2026-10-18(2).csv
  ../Resources/exchange_rates_2026-10-18(3).csv
Inserted 1000 valid daily rows into Table_exchange_rate_daily
Skipped 0 rows with invalid dates or rates
Skipped 0 rows for other currency pairs
Monthly aggregation complete:
  Month      Average Exchange Rate
  2023-05   138.3971
  2023-06   139.7493
  2023-07   141.5124
  2023-08   144.0414
  2023-09   146.8216
  2023-10   149.4429
  2023-11   152.1154
  2023-12   154.1563
  2024-01   159.7283
  2024-02   151.2907
  2024-03   137.0702
  2024-04   131.5728
  2024-05   131.6356
  2024-06   129.3439
  2024-07   129.9046
  2024-08   129.3282
  2024-09   129.2019
  2024-10   129.1962
  2024-11   129.4075
  2024-12   129.3459
  2025-01   129.3862
  2025-02   129.3043
  2025-03   129.3367
  2025-04   129.5189
  2025-05   129.2767
  2025-

In [13]:
"""
Inflation_rates.db pipeline
===========================
Stage 1 : load raw daily T-bill/inflation rate CSV -> Table_inflation_daily
Stage 2 : aggregate monthly averages by tenor      -> Table_inflation_monthly
Stage 3 : export monthly table                     -> ../Results/cleaned_inflation_rates_monthly.csv

Expected raw CSV headers:
    "Date","Tenor","Rate (%)","Source"

Expected date format:
    Jan 2, 2023

Expected output columns:
    Month,Tenor,Average_Rate
"""

import csv
import os
import sqlite3
from datetime import datetime

# ----------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------
DB_PATH = "Inflation_rates.db"


# Write the exact path to your single CSV file here.
RAW_CSV_FILE = "../Resources/bills/treasury_bills_2026-10-18.csv"

OUT_CSV = "../Results/cleaned_inflation_rates_monthly.csv"

# Optional: process only selected tenors.
# Leave as None to include every tenor found in the CSV.
SELECTED_TENORS = None
# Example:
# SELECTED_TENORS = {"91-Day", "182-Day", "364-Day"}


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------
def normalize_date(raw):
    """
    Converts dates such as 'Jan 2, 2023' to ISO format: '2023-01-02'.
    Returns None if the date cannot be parsed.
    """
    if raw is None:
        return None

    raw = str(raw).strip()

    date_formats = (
        "%b %d, %Y",     # Jan 2, 2023
        "%B %d, %Y",     # January 2, 2023
        "%m/%d/%Y",      # 01/02/2023
        "%d/%m/%Y",      # 02/01/2023
        "%Y-%m-%d",      # 2023-01-02
    )

    for fmt in date_formats:
        try:
            return datetime.strptime(raw, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue

    return None


def clean_rate(raw):
    """
    Converts values such as '9.37%', '9.37', or '1,234.56' to float.
    Returns None for missing or invalid values.
    """
    if raw is None:
        return None

    s = (
        str(raw)
        .replace("%", "")
        .replace(",", "")
        .strip()
    )

    if s == "" or s.lower() in {"nan", "none", "n/a", "-"}:
        return None

    try:
        value = float(s)
    except ValueError:
        return None

    return value


def clean_tenor(raw):
    """
    Cleans and standardizes the tenor name.
    """
    if raw is None:
        return None

    tenor = str(raw).strip()

    return tenor if tenor else None


# ----------------------------------------------------------------------
# Stage 1 : Load raw daily rate data
# ----------------------------------------------------------------------
def load_raw(cursor, conn):
    """
    Reads the raw CSV file, cleans it, and loads it into SQLite.
    """
    cursor.execute("DROP TABLE IF EXISTS Table_inflation_daily")

    cursor.execute("""
        CREATE TABLE Table_inflation_daily (
            Date TEXT PRIMARY KEY,
            Tenor TEXT,
            Rate REAL,
            Source TEXT
        )
    """)

    if not os.path.exists(RAW_CSV_FILE):
        raise FileNotFoundError(f"CSV file not found: {RAW_CSV_FILE}")

    print(f"Processing CSV file: {RAW_CSV_FILE}")

    batch = []
    skipped_invalid = 0
    skipped_tenor = 0

    with open(RAW_CSV_FILE, newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)

        for row in reader:
            date = normalize_date(row.get("Date"))
            tenor = clean_tenor(row.get("Tenor"))
            rate = clean_rate(row.get("Rate (%)"))
            source = str(row.get("Source", "")).strip()

            if date is None or tenor is None or rate is None:
                skipped_invalid += 1
                continue

            if SELECTED_TENORS and tenor not in SELECTED_TENORS:
                skipped_tenor += 1
                continue

            batch.append((
                date,
                tenor,
                rate,
                source,
            ))

    cursor.executemany(
        """
        INSERT OR REPLACE INTO Table_inflation_daily
            (Date, Tenor, Rate, Source)
        VALUES (?, ?, ?, ?)
        """,
        batch,
    )

    conn.commit()

    print(f"Inserted {len(batch)} valid daily rows into Table_inflation_daily")
    print(f"Skipped {skipped_invalid} rows with invalid dates, tenors, or rates")

    if SELECTED_TENORS:
        print(f"Skipped {skipped_tenor} rows for unselected tenors")


# ----------------------------------------------------------------------
# Stage 2 : Calculate monthly averages by tenor
# ----------------------------------------------------------------------
def aggregate_monthly(cursor, conn):
    """
    Groups daily rates by YYYY-MM and Tenor,
    then calculates the average rate for each group.
    """
    cursor.execute("DROP TABLE IF EXISTS Table_inflation_monthly")

    cursor.execute("""
        CREATE TABLE Table_inflation_monthly (
            Month TEXT,
            Tenor TEXT,
            Average_Rate REAL,
            PRIMARY KEY (Month, Tenor)
        )
    """)

    cursor.execute("""
        INSERT INTO Table_inflation_monthly
            (Month, Tenor, Average_Rate)
        SELECT
            substr(Date, 1, 7) AS Month,
            Tenor,
            ROUND(AVG(Rate), 4) AS Average_Rate
        FROM Table_inflation_daily
        GROUP BY substr(Date, 1, 7), Tenor
        ORDER BY Month, Tenor
    """)

    conn.commit()

    print("Monthly aggregation complete:")
    print("  Month      Tenor        Average Rate")

    cursor.execute("""
        SELECT Month, Tenor, Average_Rate
        FROM Table_inflation_monthly
        ORDER BY Month, Tenor
    """)

    for month, tenor, average_rate in cursor.fetchall():
        print(f"  {month}   {tenor:<12} {average_rate}")


# ----------------------------------------------------------------------
# Stage 3 : Export monthly averages to CSV
# ----------------------------------------------------------------------
def export_monthly(cursor):
    """
    Exports monthly average rates to a cleaned CSV file.
    """
    cursor.execute("""
        SELECT
            Month,
            Tenor,
            Average_Rate
        FROM Table_inflation_monthly
        ORDER BY Month, Tenor
    """)

    rows = cursor.fetchall()
    column_names = [description[0] for description in cursor.description]

    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(column_names)
        writer.writerows(rows)

    print(f"Wrote {len(rows)} monthly rows to {OUT_CSV}")


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
def main():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    print(f"Connected to database: {DB_PATH}")

    try:
        load_raw(cursor, conn)
        aggregate_monthly(cursor, conn)
        export_monthly(cursor)
    finally:
        conn.close()
        print("Database connection closed.")


if __name__ == "__main__":
    main()

Connected to database: Inflation_rates.db
Processing CSV file: ../Resources/bills/treasury_bills_2026-10-18.csv
Inserted 594 valid daily rows into Table_inflation_daily
Skipped 0 rows with invalid dates, tenors, or rates
Monthly aggregation complete:
  Month      Tenor        Average Rate
  2023-01   364-Day      10.416
  2023-02   364-Day      10.6175
  2023-03   364-Day      10.75
  2023-04   364-Day      10.9125
  2023-05   364-Day      11.282
  2023-06   364-Day      11.6925
  2023-07   364-Day      12.464
  2023-08   364-Day      13.5525
  2023-09   364-Day      14.6425
  2023-10   364-Day      15.262
  2023-11   364-Day      15.57
  2023-12   364-Day      15.8125
  2024-01   364-Day      16.356
  2024-02   364-Day      16.845
  2024-03   364-Day      16.9825
  2024-04   364-Day      16.702
  2024-05   364-Day      16.54
  2024-06   364-Day      16.735
  2024-07   364-Day      16.864
  2024-08   364-Day      16.89
  2024-09   364-Day      16.818
  2024-10   364-Day      16.11
  20